In [2]:
# parameters for the request
params = dict(
    channel="BHZ",
    starttime="1995-11-14T06:32:55.750000Z",
    endtime="1995-11-14T06:35:55.750000Z",
    level="channel",
    includeoverlaps="false",
    nodata=404,
)


In [ ]:
# Use ObsPy's RoutingClient to send the request
from obspy.clients.fdsn import RoutingClient

client = RoutingClient(
    "iris-federator",
    debug=False,
    timeout=10,
    # include_providers=["SCEDC"],
    exclude_providers=None,
    credentials=None,
)
# client.get_stations(**params)
st = client.get_waveforms(**params)


In [15]:
# Define a custom enhanced FederatorRoutingClient
import collections
from obspy.clients.fdsn.client import get_bulk_string
from obspy.clients.fdsn.routing.federator_routing_client import FederatorRoutingClient


class EnhancedFederatorRoutingClient(FederatorRoutingClient):
    """
    Enhanced FederatorRoutingClient that allows more control over the request.
    """

    def __init__(
        self, debug=False, timeout=120, include_providers=None, exclude_providers=None
    ):
        """
        Initialize the client.

        Parameters
        ----------
        debug : bool, optional
            If True, print out debug information.
        timeout : int, optional
            Timeout in seconds.

        include_providers : list of str, optional
            List of providers to include. If not given, all providers are
            included.
        exclude_providers : list of str, optional
            List of providers to exclude. If not given, no providers are
            excluded.
        """
        super().__init__(
            url="http://service.iris.edu/irisws/fedcatalog/1", 
            debug=debug,
            timeout=timeout,
            include_providers=include_providers,
            exclude_providers=exclude_providers,
        )


    def get_available_channels(self, **kwargs):
        """
        Get available channels. Accepts the same parameters as get_stations.
        """
        # convert the six parameters to a bulk. 
        bulk = [kwargs.pop(key, '*') for key in (
                "network", "station", "location", "channel", "starttime",
                "endtime")]
   
        params = collections.OrderedDict()
        for k in self.kwargs_of_interest:
            if k in kwargs:
                params[k] = str(kwargs[k])
        params["format"] = "request"

        bulk_str = get_bulk_string(bulk, params)

        
        return self.get_stations_bulk([bulk], **kwargs)


client = EnhancedFederatorRoutingClient()

# st = client.get_waveforms(**params)

# Download the routing response
r = client._download(client._url + "/query", params=params, content_type="text/plain")

# split the responses for multiple datacenters
records = client._split_routing_response(r.text, service="station")

# remove data centers that we don't need
records = client._filter_requests(records)


In [9]:
len(records)


12

In [ ]:
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import (
    FDSNNoServiceException,
    FDSNNoDataException,
    FDSNTimeoutException,
)
from pathlib import Path


Path("stations").mkdir(exist_ok=True, parents=True)
Path("mseed").mkdir(exist_ok=True, parents=True)


from multiprocessing.dummy import Pool as ThreadPool


def _try_download_bulk(datacenter, bulk):
    try:
        client = Client(datacenter, timeout=120)
    except FDSNNoServiceException:
        print("No service for", datacenter)
        return None, None
    try:
        inv = client.get_stations_bulk(bulk=records[datacenter], level="response")
        st = client.get_waveforms_bulk(bulk=records[datacenter])
        print(len(st))
        return inv, st
    except (
        FDSNNoDataException,
        FDSNTimeoutException,
        FDSNBadGatewayException,
        ValueError,
    ):
        return None, None


pool = ThreadPool(processes=len(records.keys()))
args = [(datacenter, records[datacenter]) for datacenter in records.keys()]
pool.starmap(_try_download_bulk, args)
pool.close()
